# Calgary Solar Energy Production Forecaster — Exploratory Data Analysis

This notebook explores the City of Calgary's municipal solar PV production data,
examining production distributions, seasonal patterns, and facility-level comparisons
to inform feature engineering and model design.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_and_prepare_data, compute_facility_stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries loaded successfully.")

## 1. Data Loading

Fetch and preprocess production data from Calgary Open Data (or local cache).

In [ ]:
data = load_and_prepare_data()

production = data["production"]
sites = data["sites"]
facility_stats = data["facility_stats"]

print(f"Production records: {len(production):,}")
print(f"Unique facilities: {production['facility_name'].nunique()}")
print(f"Date range: {production['period_dt'].min()} to {production['period_dt'].max()}")
print(f"\nColumns: {list(production.columns)}")

In [ ]:
production.head(10)

In [ ]:
production.info()

In [ ]:
production.describe()

## 2. Production Distributions

Examine the overall distribution of monthly solar PV production values.

In [ ]:
fig = px.histogram(
    production,
    x="solar_pv_production_kwh",
    nbins=50,
    title="Distribution of Monthly Solar PV Production (All Facilities)",
    labels={"solar_pv_production_kwh": "Monthly Production (kWh)"},
    color_discrete_sequence=["#f4a261"],
)
fig.update_layout(yaxis_title="Count")
fig.show()

In [ ]:
fig = px.box(
    production,
    x="facility_name",
    y="solar_pv_production_kwh",
    title="Production Distribution by Facility",
    labels={
        "facility_name": "Facility",
        "solar_pv_production_kwh": "Monthly Production (kWh)",
    },
    color="facility_name",
)
fig.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
    height=500,
)
fig.show()

In [ ]:
# Log-scale histogram to see spread
production["log_production"] = np.log1p(production["solar_pv_production_kwh"])

fig = px.histogram(
    production,
    x="log_production",
    nbins=40,
    title="Distribution of Log-Transformed Production",
    labels={"log_production": "log(1 + Production kWh)"},
    color_discrete_sequence=["#2a9d8f"],
)
fig.show()

## 3. Seasonal Patterns

Solar production is strongly driven by seasonal sunlight availability.
Calgary experiences dramatic variation between winter and summer months.

In [ ]:
# Average production by month across all facilities
monthly_avg = (
    production.groupby("month")["solar_pv_production_kwh"]
    .mean()
    .reset_index()
)

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]
monthly_avg["month_name"] = monthly_avg["month"].apply(lambda m: month_names[m - 1])

fig = px.bar(
    monthly_avg,
    x="month_name",
    y="solar_pv_production_kwh",
    title="Average Monthly Solar Production (All Facilities)",
    labels={
        "month_name": "Month",
        "solar_pv_production_kwh": "Avg Production (kWh)",
    },
    color="solar_pv_production_kwh",
    color_continuous_scale="YlOrRd",
)
fig.update_layout(
    xaxis=dict(categoryorder="array", categoryarray=month_names)
)
fig.show()

In [ ]:
# Seasonal heatmap: facility vs month
pivot = production.pivot_table(
    index="facility_name",
    columns="month",
    values="solar_pv_production_kwh",
    aggfunc="mean",
)
pivot.columns = month_names

fig = px.imshow(
    pivot,
    title="Average Production Heatmap: Facility x Month",
    labels=dict(x="Month", y="Facility", color="Avg kWh"),
    aspect="auto",
    color_continuous_scale="YlOrRd",
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Citywide total production over time
citywide = (
    production.groupby("period_dt")["solar_pv_production_kwh"]
    .sum()
    .reset_index()
    .sort_values("period_dt")
)

fig = px.line(
    citywide,
    x="period_dt",
    y="solar_pv_production_kwh",
    title="Citywide Total Solar Production Over Time",
    labels={
        "period_dt": "Month",
        "solar_pv_production_kwh": "Total Production (kWh)",
    },
)
fig.update_layout(hovermode="x unified")
fig.show()

In [ ]:
# Year-over-year comparison (all facilities combined)
yearly_monthly = (
    production.groupby(["year", "month"])["solar_pv_production_kwh"]
    .sum()
    .reset_index()
)
yearly_monthly["year_str"] = yearly_monthly["year"].astype(str)

fig = px.line(
    yearly_monthly,
    x="month",
    y="solar_pv_production_kwh",
    color="year_str",
    title="Year-over-Year Citywide Production",
    labels={
        "month": "Month",
        "solar_pv_production_kwh": "Total Production (kWh)",
        "year_str": "Year",
    },
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

## 4. Facility Comparison

Compare production levels, variability, and trends across facilities.

In [ ]:
# Facility summary statistics
facility_stats_display = facility_stats.sort_values("total_kwh", ascending=False)
facility_stats_display

In [ ]:
# Total production ranking
fig = px.bar(
    facility_stats_display,
    x="total_kwh",
    y="facility_name",
    orientation="h",
    title="Total Production by Facility",
    labels={
        "total_kwh": "Total Production (kWh)",
        "facility_name": "Facility",
    },
    color="total_kwh",
    color_continuous_scale="Viridis",
)
fig.show()

In [ ]:
# Production time series for all facilities
fig = px.line(
    production.sort_values("period_dt"),
    x="period_dt",
    y="solar_pv_production_kwh",
    color="facility_name",
    title="Monthly Production by Facility",
    labels={
        "period_dt": "Month",
        "solar_pv_production_kwh": "Production (kWh)",
        "facility_name": "Facility",
    },
)
fig.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.3),
    height=600,
)
fig.show()

In [ ]:
# Coefficient of variation (variability relative to mean)
facility_stats_display["cv"] = (
    facility_stats_display["std_monthly_kwh"]
    / facility_stats_display["avg_monthly_kwh"]
)

fig = px.bar(
    facility_stats_display.sort_values("cv", ascending=False),
    x="facility_name",
    y="cv",
    title="Production Variability by Facility (Coefficient of Variation)",
    labels={
        "facility_name": "Facility",
        "cv": "Coefficient of Variation",
    },
    color="cv",
    color_continuous_scale="RdYlGn_r",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Stacked area chart — all facilities
facility_monthly = (
    production.groupby(["period_dt", "facility_name"])["solar_pv_production_kwh"]
    .sum()
    .reset_index()
    .sort_values("period_dt")
)

fig = px.area(
    facility_monthly,
    x="period_dt",
    y="solar_pv_production_kwh",
    color="facility_name",
    title="Stacked Production Over Time",
    labels={
        "period_dt": "Month",
        "solar_pv_production_kwh": "Production (kWh)",
        "facility_name": "Facility",
    },
)
fig.update_layout(
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.3),
    height=600,
)
fig.show()

## 5. Feature Engineering Preview

Inspect the engineered features that will be used for modeling.

In [ ]:
# Show feature columns for one facility
sample_facility = production["facility_name"].unique()[0]
sample = production[production["facility_name"] == sample_facility].tail(24)

feature_cols = [
    "period_dt", "facility_name", "solar_pv_production_kwh",
    "month_sin", "month_cos", "year", "month",
    "rolling_avg_3m", "rolling_avg_6m", "rolling_avg_12m",
    "lag_1m", "lag_3m", "lag_12m",
]

sample[feature_cols]

In [ ]:
# Correlation matrix of numeric features
numeric_cols = [
    "solar_pv_production_kwh", "month_sin", "month_cos", "year",
    "rolling_avg_3m", "rolling_avg_6m", "rolling_avg_12m",
    "lag_1m", "lag_3m", "lag_12m",
]

corr_matrix = production[numeric_cols].corr()

fig = px.imshow(
    corr_matrix,
    title="Feature Correlation Matrix",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    text_auto=".2f",
)
fig.update_layout(height=600, width=700)
fig.show()

In [ ]:
# Rolling average vs actual for one facility
fac_data = production[production["facility_name"] == sample_facility].sort_values("period_dt")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fac_data["period_dt"],
    y=fac_data["solar_pv_production_kwh"],
    mode="lines",
    name="Actual",
    line=dict(color="#1f77b4"),
))

fig.add_trace(go.Scatter(
    x=fac_data["period_dt"],
    y=fac_data["rolling_avg_3m"],
    mode="lines",
    name="3-Month Rolling Avg",
    line=dict(color="#ff7f0e", dash="dash"),
))

fig.add_trace(go.Scatter(
    x=fac_data["period_dt"],
    y=fac_data["rolling_avg_12m"],
    mode="lines",
    name="12-Month Rolling Avg",
    line=dict(color="#2ca02c", dash="dot"),
))

fig.update_layout(
    title=f"Production with Rolling Averages — {sample_facility}",
    xaxis_title="Month",
    yaxis_title="Production (kWh)",
    hovermode="x unified",
)
fig.show()

## 6. Key Takeaways

1. **Strong seasonality**: Solar production peaks in June-July and drops to ~20-25% of peak in December-January, consistent with Calgary's northern latitude.
2. **Facility variation**: Production levels vary significantly across facilities due to different installed capacities and system configurations.
3. **Upward trend**: Year-over-year production tends to increase slightly, likely due to new installations and system optimizations.
4. **Lag features are strongly correlated**: The 1-month and 12-month lags show high correlation with current production, making them valuable predictors.
5. **Rolling averages smooth noise**: The 12-month rolling average captures the underlying capacity trend per facility.

These observations support the use of cyclical month encoding, lag features, and rolling averages for the forecasting model.